# JEDI Profile-Fit Score Regression Experiment

**Goal:** predict multiple profile-fit scores from raw job attributes.

This is the bridge between the baseline binary classifier and the later stacked model.

The experiment asks:

```text
raw job fields -> predicted profile-fit scores
```

Later, these predicted scores can be appended to the job table and used by the `applied=yes/no` classifier.

Important rule: these score models only use raw job fields. They do not use the true manually annotated score columns as input features.

## 1. Experiment Design

We try three model families:

| Model | Why we try it |
|---|---|
| Ridge Regression | strong baseline for TF-IDF + one-hot features; stable on small data |
| Random Forest Regressor | tests non-linear rules and feature interactions |
| Gradient Boosting Regressor + SVD | stronger boosted model; SVD compresses sparse text/category features before boosting |

We keep the parameter search intentionally small. The purpose is to learn whether profile-fit score prediction is promising, not to spend hours tuning.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
DATA_PATH = Path("../../data/jedi_master_prepared.csv")
DATA_PATH

WindowsPath('../../data/jedi_master_prepared.csv')

## 2. Load Dataset

The prepared dataset already contains standardized helper fields such as `role_family_standard`, `seniority_standard`, `location_bucket`, boolean city/country flags, and `skills_array`.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head(3))
display(df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}).head(60))

Rows: 331
Columns: 51


,row_id,source_platform,source_type,scraped_at,can_apply_now,company_name,company_match_score,role_title,role_title_preference_score,role_family,role_family_preference_score,seniority,seniority_profile_fit_score,location,location_profile_fit_score,work_mode,work_mode_fit_score,employment_type,experience_min_years,experience_max_years,experience_profile_fit_score,skills_or_requirements,skills_profile_fit_score,salary_min,salary_max,salary_profile_fit_score,freshness,overall_preference_score,preference_reason,applied,apply_url,skills_or_requirements_original,skills_array,has_skills_or_requirements,skills_imputation_type,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,is_bengaluru,is_gurgaon_ncr,is_india,experience_min_bucket,salary_midpoint,salary_available,near_duplicate_key,company_name_standard,role_title_standard
0,1,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Nirmata,4,"Software Engineer Intern, India",5,Backend / Cloud,4,intern,5,"Bangalore, Hybrid",4,Hybrid,5,internship,0.0,1.0,5.0,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,2,1200000,1200000,5,Not visible,3,"Intern role is fine, everything is good but the requirements state Kubernetes, observability tools, promethus and Go...",no,https://job-boards.greenhouse.io/nirmata/jobs/5040355008,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,"[""go"",""java"",""microservices"",""kubernetes"",""prometheus"",""timescale"",""reliability"",""automation""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,1200000,True,nirmata|software_engineer_intern_india|bengaluru,nirmata,software_engineer_intern_india
1,2,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,CloudSEK,5,SDE - Backend - Intern,5,Backend,5,intern,5,"Bengaluru, Karnataka, India",4,Hybrid,5,internship,0.0,1.0,5.0,Backend engineering; cybersecurity product; APIs; databases likely,4,800000,900000,4,Not visible,4,"Backend intern in Bengaluru; very relevant for fresher profile, salary is a little less and location bengaluru",yes,https://job-boards.greenhouse.io/cloudsek/jobs/5766402004,Backend engineering; cybersecurity product; APIs; databases likely,"[""backend engineering"",""cybersecurity product"",""apis"",""databases likely""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,850000,True,cloudsek|sde_backend_intern|bengaluru,cloudsek,sde_backend_intern
2,3,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Anaplan,5,Associate Software Engineer,5,Backend / Fullstack,5,entry_level,5,"Gurugram, India",5,unknown,5,full_time,0.0,2.0,5.0,Software engineering; AI-infused platform; SaaS; cloud; planning systems,5,500000,700000,2,Not visible,3,Associate level in Gurgaon; high-quality company and location fitbut salary is quite less and the bad factor,yes,https://job-boards.greenhouse.io/anaplan/jobs/8511489002,Software engineering; AI-infused platform; SaaS; cloud; planning systems,"[""software engineering"",""ai-infused platform"",""saas"",""cloud"",""planning systems""]",True,original,fullstack,entry_level,full_time,unknown,delhi_ncr,india,delhi_ncr,False,True,True,0,600000,True,anaplan|associate_software_engineer|delhi_ncr,anaplan,associate_software_engineer


,column,dtype
0,row_id,int64
1,source_platform,str
2,source_type,str
3,scraped_at,str
4,can_apply_now,str
5,company_name,str
6,company_match_score,int64
7,role_title,str
8,role_title_preference_score,int64
9,role_family,str


## 3. Choose Regression Targets

Default target set:

- `overall_preference_score`
- `role_title_preference_score`
- `role_family_preference_score`
- `seniority_profile_fit_score`
- `location_profile_fit_score`
- `skills_profile_fit_score`

We skip salary/work-mode/company for the first score experiment because those are either saturated or less useful right now. You can add them later if needed.

In [3]:
TARGET_COLUMNS = [
    "overall_preference_score",
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "skills_profile_fit_score",
]

OPTIONAL_TARGET_COLUMNS = [
    "company_match_score",
    "work_mode_fit_score",
    "salary_profile_fit_score",
]

missing_targets = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

target_audit = pd.DataFrame({
    "target": TARGET_COLUMNS,
    "missing": [df[c].isna().sum() for c in TARGET_COLUMNS],
    "unique_values": [df[c].nunique(dropna=True) for c in TARGET_COLUMNS],
    "mean": [df[c].mean() for c in TARGET_COLUMNS],
    "std": [df[c].std() for c in TARGET_COLUMNS],
    "min": [df[c].min() for c in TARGET_COLUMNS],
    "max": [df[c].max() for c in TARGET_COLUMNS],
})

display(target_audit)
display(df[TARGET_COLUMNS].describe().T)

,target,missing,unique_values,mean,std,min,max
0,overall_preference_score,0,6,3.099698,1.413903,0,5
1,role_title_preference_score,0,6,3.963746,1.430792,0,5
2,role_family_preference_score,0,5,4.441088,0.930300,1,5
3,seniority_profile_fit_score,0,6,3.486405,1.485984,0,5
4,location_profile_fit_score,0,6,3.199396,1.644863,0,5
5,skills_profile_fit_score,0,5,4.096677,1.019368,1,5


,count,mean,std,min,25%,50%,75%,max
overall_preference_score,331.0,3.099698,1.413903,0.0,2.0,3.0,4.0,5.0
role_title_preference_score,331.0,3.963746,1.430792,0.0,3.0,5.0,5.0,5.0
role_family_preference_score,331.0,4.441088,0.930300,1.0,4.0,5.0,5.0,5.0
seniority_profile_fit_score,331.0,3.486405,1.485984,0.0,2.0,4.0,5.0,5.0
location_profile_fit_score,331.0,3.199396,1.644863,0.0,1.0,4.0,4.0,5.0
skills_profile_fit_score,331.0,4.096677,1.019368,1.0,4.0,4.0,5.0,5.0


## 4. Feature Policy

We remove all human score columns from the input features. This prevents the model from learning one score by looking at another manually assigned score.

We also remove final action/leakage columns such as `applied`, `preference_reason`, `apply_url`, source columns, identifiers, and duplicate keys.

In [4]:
ALL_SCORE_COLUMNS = [c for c in df.columns if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]]

LEAKAGE_OR_METADATA_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_OR_METADATA_COLUMNS))
available_drop_columns = [c for c in drop_columns if c in df.columns]
candidate_features = [c for c in df.columns if c not in available_drop_columns]

display(pd.DataFrame({"dropped_column": available_drop_columns}))
display(pd.DataFrame({"candidate_feature": candidate_features}))

,dropped_column
0,applied
1,apply_url
2,company_match_score
3,company_name_standard
4,experience_profile_fit_score
5,location_profile_fit_score
6,near_duplicate_key
7,overall_preference_score
8,preference_reason
9,role_family_preference_score


,candidate_feature
0,can_apply_now
1,company_name
2,role_title
3,role_family
4,seniority
5,location
6,work_mode
7,employment_type
8,experience_min_years
9,experience_max_years


## 5. Build Modeling Table

Text fields are joined into one `job_text` column. Categorical columns are one-hot encoded. Numeric and boolean columns are imputed and scaled.

The target `y` is a multi-column dataframe with one score column per profile-fit target.

In [5]:
text_source_columns = [
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "skills_or_requirements",
    "skills_array",
]

categorical_features = [
    "can_apply_now",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
    "freshness",
]

numeric_features = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

required_inputs = text_source_columns + categorical_features + numeric_features + boolean_features
missing_inputs = [c for c in required_inputs if c not in df.columns]
if missing_inputs:
    raise ValueError(f"Missing feature columns: {missing_inputs}")

model_df = df.copy()
for col in text_source_columns:
    model_df[col] = model_df[col].fillna("").astype(str)

model_df["job_text"] = model_df[text_source_columns].agg(" ".join, axis=1)

for col in boolean_features:
    model_df[col] = model_df[col].fillna(False).astype(int)

feature_columns = ["job_text"] + categorical_features + numeric_features + boolean_features

valid_target_rows = model_df[TARGET_COLUMNS].notna().all(axis=1)
X = model_df.loc[valid_target_rows, feature_columns].copy()
y = model_df.loc[valid_target_rows, TARGET_COLUMNS].copy().astype(float)

# Keep row ids for later export. If row_id ever disappears, use dataframe index.
row_ids = model_df.loc[valid_target_rows, "row_id"] if "row_id" in model_df.columns else model_df.loc[valid_target_rows].index.to_series()

print(f"Rows used for regression: {len(X):,}")
print(f"Input columns: {len(feature_columns)}")
display(X.head(3))
display(y.head(3))

Rows used for regression: 331
Input columns: 21


,job_text,can_apply_now,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,experience_min_bucket,skills_imputation_type,freshness,experience_min_years,salary_min,salary_max,salary_midpoint,has_skills_or_requirements,is_bengaluru,is_gurgaon_ncr,is_india,salary_available
0,"Nirmata Software Engineer Intern, India Backend / Cloud intern Bangalore, Hybrid Go; Java; microservices; Kubernetes...",Yes,backend,intern,internship,hybrid,bengaluru,india,bengaluru,0,original,Not visible,0.0,1200000,1200000,1200000,1,1,0,1,1
1,"CloudSEK SDE - Backend - Intern Backend intern Bengaluru, Karnataka, India Backend engineering; cybersecurity produc...",Yes,backend,intern,internship,hybrid,bengaluru,india,bengaluru,0,original,Not visible,0.0,800000,900000,850000,1,1,0,1,1
2,"Anaplan Associate Software Engineer Backend / Fullstack entry_level Gurugram, India Software engineering; AI-infused...",Yes,fullstack,entry_level,full_time,unknown,delhi_ncr,india,delhi_ncr,0,original,Not visible,0.0,500000,700000,600000,1,0,1,1,1


,overall_preference_score,role_title_preference_score,role_family_preference_score,seniority_profile_fit_score,location_profile_fit_score,skills_profile_fit_score
0,3.0,5.0,4.0,5.0,4.0,2.0
1,4.0,5.0,5.0,5.0,4.0,4.0
2,3.0,5.0,5.0,5.0,5.0,5.0


## 6. Train/Test Split

The score experiment uses a train/test split for final inspection and k-fold cross-validation for model comparison.

If `applied` exists, we stratify the split by the binary action label so the holdout set has similar yes/no composition.

In [6]:
stratify_labels = None
if "applied" in model_df.columns:
    stratify_labels = model_df.loc[valid_target_rows, "applied"].astype(str).str.lower().str.strip()

X_train, X_test, y_train, y_test, row_ids_train, row_ids_test = train_test_split(
    X,
    y,
    row_ids,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_labels,
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")
display(y_train.describe().T[["mean", "std", "min", "max"]])

Train rows: 264
Test rows:  67


,mean,std,min,max
overall_preference_score,3.041667,1.414942,0.0,5.0
role_title_preference_score,3.893939,1.470923,0.0,5.0
role_family_preference_score,4.386364,0.960416,1.0,5.0
seniority_profile_fit_score,3.439394,1.511716,0.0,5.0
location_profile_fit_score,3.132576,1.666443,0.0,5.0
skills_profile_fit_score,4.064394,1.049905,1.0,5.0


## 7. Preprocessing Blocks

Ridge and Random Forest use the direct sparse preprocessing output. Gradient Boosting uses the same preprocessing and then compresses it with SVD, because classic gradient boosting expects dense numeric input and does not behave well on huge sparse TF-IDF matrices.

In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
])

text_transformer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=1200,
)

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features + boolean_features),
        ("categorical", categorical_transformer, categorical_features),
        ("text", text_transformer, "job_text"),
    ]
)

preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

## 8. Metrics For Multi-Output Score Regression

Accuracy is not the right metric for a 0-5 score. We evaluate:

- `MAE`: average absolute score error. Lower is better.
- `RMSE`: punishes large mistakes more strongly.
- `R2`: explained variance. Useful, but unstable on small datasets.
- `Spearman`: ranking agreement. Useful because we care about ranking jobs.
- `within_0_5`: share of predictions within half a score point.
- `within_1_0`: share of predictions within one score point.

For a 0-5 score, rough interpretation:

```text
MAE <= 0.60: strong
MAE 0.60-0.90: useful
MAE > 1.00: weak for that target
```

In [8]:
def clip_scores(predictions):
    return np.clip(np.asarray(predictions, dtype=float), 0.0, 5.0)


def regression_metric_table(y_true, y_pred, targets):
    y_true_df = pd.DataFrame(np.asarray(y_true, dtype=float), columns=targets)
    y_pred_df = pd.DataFrame(clip_scores(y_pred), columns=targets)

    rows = []
    for target in targets:
        actual = y_true_df[target]
        pred = y_pred_df[target]
        error = pred - actual
        abs_error = error.abs()

        rows.append({
            "target": target,
            "mae": abs_error.mean(),
            "rmse": np.sqrt(np.mean(np.square(error))),
            "r2": r2_score(actual, pred),
            "spearman": actual.corr(pred, method="spearman"),
            "within_0_5": (abs_error <= 0.5).mean(),
            "within_1_0": (abs_error <= 1.0).mean(),
        })

    return pd.DataFrame(rows)


def summarize_metric_table(metric_table):
    return {
        "macro_mae": metric_table["mae"].mean(),
        "macro_rmse": metric_table["rmse"].mean(),
        "macro_r2": metric_table["r2"].mean(),
        "macro_spearman": metric_table["spearman"].mean(),
        "macro_within_0_5": metric_table["within_0_5"].mean(),
        "macro_within_1_0": metric_table["within_1_0"].mean(),
    }


def evaluate_predictions(model_name, params_label, split_name, y_true, y_pred):
    per_target = regression_metric_table(y_true, y_pred, TARGET_COLUMNS)
    summary = summarize_metric_table(per_target)
    summary.update({
        "model": model_name,
        "params": params_label,
        "split": split_name,
    })
    return summary, per_target

## 9. Define Candidate Models

Small search grid only:

- Ridge: few `alpha` values.
- Random Forest: depth/leaf variations.
- Gradient Boosting + SVD: few tree-depth/learning-rate/estimator-count variations.

In [9]:
def make_ridge(alpha):
    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("regressor", Ridge(alpha=alpha, random_state=RANDOM_STATE)),
    ])


def make_random_forest(n_estimators, max_depth, min_samples_leaf):
    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("regressor", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


def make_gradient_boosting_svd(n_components, n_estimators, learning_rate, max_depth, subsample):
    base = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("svd", TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)),
        ("regressor", MultiOutputRegressor(base)),
    ])


candidate_models = [
    ("Ridge", "alpha=0.5", make_ridge(alpha=0.5)),
    ("Ridge", "alpha=1.0", make_ridge(alpha=1.0)),
    ("Ridge", "alpha=3.0", make_ridge(alpha=3.0)),
    ("Ridge", "alpha=10.0", make_ridge(alpha=10.0)),

    ("Random Forest", "trees=250 depth=None leaf=2", make_random_forest(250, None, 2)),
    ("Random Forest", "trees=250 depth=10 leaf=2", make_random_forest(250, 10, 2)),
    ("Random Forest", "trees=350 depth=8 leaf=3", make_random_forest(350, 8, 3)),

    ("Gradient Boosting + SVD", "svd=60 trees=100 lr=0.05 depth=2", make_gradient_boosting_svd(60, 100, 0.05, 2, 0.85)),
    ("Gradient Boosting + SVD", "svd=60 trees=150 lr=0.05 depth=2", make_gradient_boosting_svd(60, 150, 0.05, 2, 0.85)),
    ("Gradient Boosting + SVD", "svd=80 trees=120 lr=0.03 depth=3", make_gradient_boosting_svd(80, 120, 0.03, 3, 0.85)),
]

pd.DataFrame([{"model": m, "params": p} for m, p, _ in candidate_models])

,model,params
0,Ridge,alpha=0.5
1,Ridge,alpha=1.0
2,Ridge,alpha=3.0
3,Ridge,alpha=10.0
4,Random Forest,trees=250 depth=None leaf=2
5,Random Forest,trees=250 depth=10 leaf=2
6,Random Forest,trees=350 depth=8 leaf=3
7,Gradient Boosting + SVD,svd=60 trees=100 lr=0.05 depth=2
8,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2
9,Gradient Boosting + SVD,svd=80 trees=120 lr=0.03 depth=3


## 10. Cross-Validated Model Comparison

This is the main comparison table. It uses out-of-fold predictions for every row. Lower `macro_mae` is better.

This cell may take a few minutes because Gradient Boosting trains one model per target per fold.

In [10]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_summaries = []
cv_per_target_tables = []
cv_predictions_by_candidate = {}

for model_name, params_label, estimator in candidate_models:
    print(f"Running CV: {model_name} | {params_label}")
    oof_pred = cross_val_predict(estimator, X, y, cv=cv, n_jobs=None)
    oof_pred = clip_scores(oof_pred)

    summary, per_target = evaluate_predictions(model_name, params_label, "cv_oof", y, oof_pred)
    cv_summaries.append(summary)
    per_target.insert(0, "params", params_label)
    per_target.insert(0, "model", model_name)
    cv_per_target_tables.append(per_target)
    cv_predictions_by_candidate[(model_name, params_label)] = oof_pred

cv_summary_df = pd.DataFrame(cv_summaries).sort_values("macro_mae", ascending=True).reset_index(drop=True)
cv_per_target_df = pd.concat(cv_per_target_tables, ignore_index=True)

display(cv_summary_df)

Running CV: Ridge | alpha=0.5
Running CV: Ridge | alpha=1.0
Running CV: Ridge | alpha=3.0
Running CV: Ridge | alpha=10.0
Running CV: Random Forest | trees=250 depth=None leaf=2
Running CV: Random Forest | trees=250 depth=10 leaf=2
Running CV: Random Forest | trees=350 depth=8 leaf=3
Running CV: Gradient Boosting + SVD | svd=60 trees=100 lr=0.05 depth=2
Running CV: Gradient Boosting + SVD | svd=60 trees=150 lr=0.05 depth=2
Running CV: Gradient Boosting + SVD | svd=80 trees=120 lr=0.03 depth=3


,macro_mae,macro_rmse,macro_r2,macro_spearman,macro_within_0_5,macro_within_1_0,model,params,split
0,0.490400,0.720061,0.648911,0.790338,0.652568,0.863545,Ridge,alpha=1.0,cv_oof
1,0.492563,0.729567,0.640938,0.784196,0.645519,0.858510,Ridge,alpha=0.5,cv_oof
2,0.502529,0.721207,0.646209,0.794610,0.627392,0.859013,Ridge,alpha=3.0,cv_oof
3,0.546433,0.758820,0.610026,0.783312,0.592145,0.845418,Ridge,alpha=10.0,cv_oof
4,0.550326,0.801204,0.565694,0.726574,0.609265,0.838872,Random Forest,trees=250 depth=None leaf=2,cv_oof
5,0.558803,0.805402,0.561505,0.720487,0.601208,0.840383,Random Forest,trees=250 depth=10 leaf=2,cv_oof
6,0.588954,0.827238,0.531979,0.731631,0.579053,0.816717,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,cv_oof
7,0.589243,0.827310,0.537213,0.702160,0.568983,0.831823,Random Forest,trees=350 depth=8 leaf=3,cv_oof
8,0.595073,0.827404,0.537234,0.730959,0.577039,0.818731,Gradient Boosting + SVD,svd=80 trees=120 lr=0.03 depth=3,cv_oof
9,0.622089,0.851294,0.510252,0.717377,0.540785,0.803625,Gradient Boosting + SVD,svd=60 trees=100 lr=0.05 depth=2,cv_oof


## 11. Best Model Per Family

This view picks the best candidate for each model family by cross-validated macro MAE.

In [11]:
best_by_family = (
    cv_summary_df
    .sort_values("macro_mae", ascending=True)
    .groupby("model", as_index=False)
    .head(1)
    .sort_values("macro_mae", ascending=True)
    .reset_index(drop=True)
)

display(best_by_family)

best_model_name = cv_summary_df.loc[0, "model"]
best_params_label = cv_summary_df.loc[0, "params"]
print(f"Best overall by CV macro MAE: {best_model_name} | {best_params_label}")

,macro_mae,macro_rmse,macro_r2,macro_spearman,macro_within_0_5,macro_within_1_0,model,params,split
0,0.490400,0.720061,0.648911,0.790338,0.652568,0.863545,Ridge,alpha=1.0,cv_oof
1,0.550326,0.801204,0.565694,0.726574,0.609265,0.838872,Random Forest,trees=250 depth=None leaf=2,cv_oof
2,0.588954,0.827238,0.531979,0.731631,0.579053,0.816717,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,cv_oof


Best overall by CV macro MAE: Ridge | alpha=1.0


## 12. Per-Target Metrics For Best Candidates

This tells us which score targets are learnable and which are weak.

Look especially at:

- high MAE targets: weakly learned score dimensions
- low Spearman targets: poor ranking signal
- low within-one-point targets: unreliable score predictions

In [12]:
best_keys = set(zip(best_by_family["model"], best_by_family["params"]))
best_per_target = cv_per_target_df[
    cv_per_target_df.apply(lambda row: (row["model"], row["params"]) in best_keys, axis=1)
].sort_values(["model", "mae"])

display(best_per_target)

,model,params,target,mae,rmse,r2,spearman,within_0_5,within_1_0
52,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,location_profile_fit_score,0.447928,0.663637,0.836727,0.894181,0.673716,0.891239
51,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,seniority_profile_fit_score,0.498765,0.717741,0.765997,0.889655,0.625378,0.854985
50,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,role_family_preference_score,0.522367,0.808153,0.243071,0.566117,0.685801,0.858006
48,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,overall_preference_score,0.625353,0.827110,0.656757,0.828726,0.486405,0.818731
53,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,skills_profile_fit_score,0.626001,0.888370,0.238201,0.580867,0.577039,0.818731
49,Gradient Boosting + SVD,svd=60 trees=150 lr=0.05 depth=2,role_title_preference_score,0.813311,1.058418,0.451122,0.630241,0.425982,0.658610
27,Random Forest,trees=250 depth=None leaf=2,seniority_profile_fit_score,0.420179,0.682420,0.788461,0.893759,0.691843,0.864048
26,Random Forest,trees=250 depth=None leaf=2,role_family_preference_score,0.523471,0.809799,0.239984,0.528788,0.673716,0.873112
28,Random Forest,trees=250 depth=None leaf=2,location_profile_fit_score,0.561312,0.754305,0.789065,0.808703,0.540785,0.836858
24,Random Forest,trees=250 depth=None leaf=2,overall_preference_score,0.576118,0.781154,0.693840,0.834198,0.558912,0.824773


## 13. Fit Best Overall Model On Train, Evaluate On Test

The CV table is the main model-selection result. This holdout evaluation is for a concrete test-set prediction file and manual inspection.

In [13]:
candidate_lookup = {(model_name, params_label): estimator for model_name, params_label, estimator in candidate_models}
best_estimator = clone(candidate_lookup[(best_model_name, best_params_label)])

best_estimator.fit(X_train, y_train)
test_pred = clip_scores(best_estimator.predict(X_test))

test_summary, test_per_target = evaluate_predictions(best_model_name, best_params_label, "holdout_test", y_test, test_pred)

display(pd.DataFrame([test_summary]))
display(test_per_target.sort_values("mae"))

,macro_mae,macro_rmse,macro_r2,macro_spearman,macro_within_0_5,macro_within_1_0,model,params,split
0,0.472369,0.663097,0.583838,0.765382,0.644279,0.885572,Ridge,alpha=1.0,holdout_test


,target,mae,rmse,r2,spearman,within_0_5,within_1_0
4,location_profile_fit_score,0.346307,0.513606,0.887180,0.905892,0.791045,0.940299
3,seniority_profile_fit_score,0.348819,0.473598,0.879562,0.940977,0.731343,0.985075
0,overall_preference_score,0.450568,0.601911,0.811503,0.907813,0.626866,0.910448
2,role_family_preference_score,0.468073,0.635185,0.308728,0.586283,0.597015,0.850746
5,skills_profile_fit_score,0.488421,0.711309,0.343568,0.676867,0.656716,0.865672
1,role_title_preference_score,0.732024,1.042971,0.272485,0.574462,0.462687,0.761194


## 14. Inspect Test Predictions

This table is for human error analysis. Focus on the rows where predicted overall score is far away from the actual score.

In [14]:
test_review = df.loc[row_ids_test.index, [
    "row_id",
    "company_name",
    "role_title",
    "role_family_standard",
    "seniority_standard",
    "location_bucket",
    "work_mode_standard",
    "skills_or_requirements",
    "applied",
]].copy()

for idx, target in enumerate(TARGET_COLUMNS):
    test_review[f"actual_{target}"] = y_test[target].values
    test_review[f"pred_{target}"] = test_pred[:, idx]

test_review["overall_abs_error"] = (
    test_review["pred_overall_preference_score"] - test_review["actual_overall_preference_score"]
).abs()

display(test_review.sort_values("overall_abs_error", ascending=False).head(20))

,row_id,company_name,role_title,role_family_standard,seniority_standard,location_bucket,work_mode_standard,skills_or_requirements,applied,actual_overall_preference_score,pred_overall_preference_score,actual_role_title_preference_score,pred_role_title_preference_score,actual_role_family_preference_score,pred_role_family_preference_score,actual_seniority_profile_fit_score,pred_seniority_profile_fit_score,actual_location_profile_fit_score,pred_location_profile_fit_score,actual_skills_profile_fit_score,pred_skills_profile_fit_score,overall_abs_error
33,34,Point72,"Machine Learning Engineer, GenAI Technology",ml_ai,lead,bengaluru,unknown,GenAI; ML engineering; financial technology; applied AI systems,no,0.0,1.913805,4.0,1.134127,4.0,2.837175,0.0,0.127216,4.0,4.014530,4.0,2.627736,1.913805
32,33,Fluxon,Senior Software Engineer (AI Engineer),backend,senior,india_remote,remote,Generative AI; RAG; LLM applications; AI engineering; backend systems,no,0.0,1.619631,2.0,2.457056,4.0,3.911820,2.0,1.210862,5.0,4.346261,4.0,3.086066,1.619631
60,61,Whoop,"Software Engineer I (Frontend, Growth)",frontend,entry_level,us,onsite,"React, JavaScript, A/B tests, frontend, analytics",yes,4.0,2.792742,5.0,4.545823,5.0,4.587073,5.0,4.887056,2.0,1.135456,4.0,3.845183,1.207258
82,83,AHEAD,Full Stack Engineer- IAM Solutions,fullstack,unknown,india_remote,remote,"Fullstack, IAM, automation, access systems, integrations",no,5.0,3.795977,5.0,3.855429,5.0,4.258606,4.0,4.003657,5.0,5.000000,5.0,4.181338,1.204023
242,243,"Pulse Labs AI, Inc.",Senior Backend Engineer,backend,senior,india_remote,remote,backend fundamentals; rest apis; databases; sql; git; debugging; llms; prompt engineering; ai product integration; a...,no,3.0,1.815702,2.0,1.851367,5.0,4.164774,2.0,1.195158,5.0,4.940986,4.0,4.034725,1.184298
289,290,TrueBuilt Software,Full-Stack Engineer,fullstack,junior,india_remote,remote,Fullstack; mobile app; web app; backend services; offline sync; integrations,yes,5.0,3.915065,5.0,4.756041,5.0,5.000000,5.0,4.128340,4.0,5.000000,5.0,4.191985,1.084935
231,232,"Clerky, Inc.",Customer Operations & Writing Specialist,other,unknown,global_remote,remote,software engineering fundamentals; problem solving; git; debugging; communication; llms; prompt engineering; ai prod...,no,2.0,2.958577,1.0,2.930962,1.0,1.619135,4.0,3.480604,4.0,3.666237,1.0,3.203569,0.958577
50,51,Netomi,SDE I (FullStack),fullstack,entry_level,india_remote,remote,"Node.js, React/Vue, REST APIs, AWS, agentic AI",yes,5.0,4.077333,5.0,4.172301,5.0,4.815457,5.0,4.549216,5.0,4.996252,5.0,4.666539,0.922667
4,5,Energy Exemplar,Associate Software Engineer - Python,backend,entry_level,bengaluru,unknown,Python; algorithms; product development; possible data/optimization systems,yes,5.0,4.105275,5.0,4.453000,5.0,4.202304,5.0,4.923446,4.0,4.706937,5.0,4.027677,0.894725
278,279,EarnIn,Senior AI Dev Platform Engineer,devops_cloud,senior,bengaluru,onsite,AI developer platform; tooling; backend services; ML/AI infrastructure; platform engineering,no,2.0,2.882324,2.0,2.588192,2.0,3.364461,2.0,1.061798,4.0,3.534283,4.0,3.413392,0.882324


## 15. Save Comparison Tables And Predictions

Exports created:

- model comparison table
- per-target metric table
- test prediction review table
- out-of-fold predicted profile scores for stacking

In [15]:
OUTPUT_DIR = Path("../../data")

comparison_path = OUTPUT_DIR / "jedi_profile_score_regression_model_comparison.csv"
per_target_path = OUTPUT_DIR / "jedi_profile_score_regression_per_target_metrics.csv"
test_predictions_path = OUTPUT_DIR / "jedi_profile_score_regression_test_predictions.csv"
oof_predictions_path = OUTPUT_DIR / "jedi_profile_score_oof_predictions.csv"

cv_summary_df.to_csv(comparison_path, index=False)
cv_per_target_df.to_csv(per_target_path, index=False)
test_review.sort_values("overall_abs_error", ascending=False).to_csv(test_predictions_path, index=False)

best_oof_pred = cv_predictions_by_candidate[(best_model_name, best_params_label)]
oof_score_features = pd.DataFrame({"row_id": row_ids.values})
for idx, target in enumerate(TARGET_COLUMNS):
    oof_score_features[f"pred_{target}"] = best_oof_pred[:, idx]
    oof_score_features[f"actual_{target}"] = y[target].values

oof_score_features["score_model"] = best_model_name
oof_score_features["score_model_params"] = best_params_label
oof_score_features.to_csv(oof_predictions_path, index=False)

print(f"Saved: {comparison_path}")
print(f"Saved: {per_target_path}")
print(f"Saved: {test_predictions_path}")
print(f"Saved: {oof_predictions_path}")

Saved: ..\..\data\jedi_profile_score_regression_model_comparison.csv
Saved: ..\..\data\jedi_profile_score_regression_per_target_metrics.csv
Saved: ..\..\data\jedi_profile_score_regression_test_predictions.csv
Saved: ..\..\data\jedi_profile_score_oof_predictions.csv


## 16. Interpretation Checklist After Running

After the notebook runs, interpret it in this order:

1. Which model has the lowest `macro_mae`?
2. Does the best model beat Ridge meaningfully? If not, prefer Ridge because it is simpler.
3. Which target has the worst MAE?
4. Is `overall_preference_score` learnable enough for ranking?
5. Are predictions within 1 point for most rows?
6. Does the error table reveal annotation inconsistency or missing feature signal?
7. Should we use the out-of-fold predicted scores in the next stacked classifier notebook?

Decision rule:

```text
If profile-score predictions have useful MAE and ranking correlation, proceed to stacking.
If not, collect more labels or simplify the score targets before stacking.
```